# SymFormer Training Notebook

Universal Symmetry-Aware Medical Segmentation

This notebook imports all necessary modules and runs training via `scripts/train.py`.

In [ ]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    
print(f"Project root: {project_root}")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## 1. Imports — All Modules

In [ ]:
# Config
from configs.config import TrainingConfig, load_config

# Datasets
from datasets.base import BaseDataset
from datasets.cpaisd import CPAISDDataset
from datasets.cpaisd_enhanced import CPAISDEnhancedDataset
from datasets.brats import BraTSDataset
from datasets.factory import get_dataset_class

# Models
from models.symformer import SymFormer
from models.conditioned_symformer import ConditionedSymFormer
from models.components import AlignmentNetwork, EncoderBlock3D, AdaptiveFusion
from models.losses import StrokeLoss, TverskyLoss, SymFormerLoss, create_ablation_losses
from models.bottleneck import get_bottleneck, BaseBottleneck, SymmetryAwareBottleneck, MambaBottleneckWrapper
from models.decoder.hvt import HVTDecoder, DecoderBlock, kMaXBlock
from models.decoder.kan import KANDecoderHead, KANHVTDecoder, EfficientKANLayer, RationalKANLayer
from models.layers.conditioning import ClinicalConditionEncoder, ConditionalCrossAttention

# Training
from training.trainer import SymFormerTrainer

# Evaluation
from evaluation.evaluator import Evaluator
from evaluation.metrics import MetricCalculator
from evaluation.complexity import get_model_complexity

# Utils
from utils.data_utils import compute_class_weights
from utils.transforms import apply_aligned_mask

print("All imports successful ✅")
print(f"Torch CUDA devices: {torch.cuda.device_count()}")

## 2. Quick Model Sanity Check

Verify forward pass before launching training.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = ConditionedSymFormer(
    in_channels=1,
    num_classes=3,
    T=1,
    input_size=(512, 512),
    bottleneck_type='mamba',
    use_kan=True,
).to(device)

x = torch.randn(2, 1, 512, 512).to(device)
meta = {
    'nihss': torch.tensor([10.0, 5.0], device=device),
    'age': torch.tensor([65.0, 70.0], device=device),
    'sex': torch.tensor([0, 1], device=device),
    'time': torch.tensor([3.0, 6.0], device=device),
    'dsa': torch.tensor([0, 1], device=device),
}

with torch.no_grad():
    out = model(x, metadata_dict=meta)
    print(f"Prediction shape: {out['pred'].shape}")
    print(f"Multiscale outputs: {len(out.get('multiscale_preds', []))}")
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

# Quick loss check
criterion = StrokeLoss(num_classes=3)
target = torch.randint(0, 3, (2, 512, 512), device=device)
loss, loss_dict = criterion(out['pred'], target)
print(f"StrokeLoss: {loss.item():.4f}")
for k, v in loss_dict.items():
    print(f"  {k}: {v:.4f}")

print("\nSanity check passed ✅")

## 3. Load Config & DataLoaders

In [ ]:
# ── Pick your dataset ──
DATASET = "cpaisd"  # Options: cpaisd, cpaisd_enhanced, brats, rsna
DEVICES = "0"       # GPU(s), e.g. "0", "1", "0,1"

os.environ["CUDA_VISIBLE_DEVICES"] = DEVICES
print(f"CUDA_VISIBLE_DEVICES={DEVICES}")

config = load_config(DATASET)
config.create_directories()
config.print_config()

In [ ]:
from torch.utils.data import DataLoader

dataset_class = get_dataset_class(config.DATASET_NAME)
root_path = config.DATA_PATHS.get(config.DATASET_NAME, config.BASE_PATH)

train_dataset = dataset_class(
    dataset_root=root_path, split='train',
    T=config.T, config=config,
)

val_dataset = dataset_class(
    dataset_root=root_path, split='val',
    T=config.T, config=config,
)

train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE,
    shuffle=True, num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE,
    shuffle=False, num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

## 4. Init Model & Trainer

In [ ]:
device_count = torch.cuda.device_count()
device = torch.device('cuda:0' if device_count > 0 else 'cpu')
multi_gpu = device_count > 1

model = ConditionedSymFormer(
    in_channels=config.NUM_CHANNELS,
    num_classes=config.NUM_CLASSES,
    T=config.T,
    input_size=config.IMAGE_SIZE,
    bottleneck_type='mamba' if config.USE_MAMBA else 'symmetry',
    use_kan=config.USE_KAN,
)

if multi_gpu:
    model = torch.nn.DataParallel(model)

trainer = SymFormerTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    multi_gpu=multi_gpu,
)

print(f"Device: {device}  Multi-GPU: {multi_gpu}")

## 5. Launch Training

Set `NUM_EPOCHS` below to control training length.

In [ ]:
# ── Override epochs (optional) ──
NUM_EPOCHS = config.NUM_EPOCHS  # default from config
# NUM_EPOCHS = 5                  # uncomment for a quick test run

trainer.train(num_epochs=NUM_EPOCHS)

## 6. Quick Evaluation

In [ ]:
from evaluation.evaluator import Evaluator

evaluator = Evaluator(
    model=model,
    val_loader=val_loader,
    device=device,
    config=config,
    num_samples=5,
    output_dir='./evaluation_results',
)
summary = evaluator.run()
print(summary)

---

### Alternative: Run `scripts/train.py` directly

Uncomment the cell below to run the CLI entry point instead of the step-by-step flow above.

In [ ]:
# %run ../scripts/train.py --devices "0" --dataset cpaisd
pass